# SQLDB

将关系数据库中的表和字段映射为因子表与因子。适合已有 SQL 数据源的读取，以及具备写权限时的因子落库。

使用流程为“配置连接 → 查询 `TableNames` 和 `FactorNames` → 选定表类型 → 按日期和证券读取”。下文保留宽表、窄表、特征表、时序表和映射表的示例；单因子返回 `DataFrame`，多因子返回 `Panel`。

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

## 配置连接

在使用者自己的 `SQLDBConfig.json` 中设置数据库类型、主机、端口、数据库名、账号与密码；`SQLDB(config_file=...)` 可指定其他文件。未指定时按用户配置目录的约定查找文件，参数优先级见 [通则和约定](../通则和约定.ipynb)。

读取需要目标表的查询权限。连接时会维护元数据侧表，写入和变更示例需要相应的建表、更新与删除权限。本篇具体表名与字段属于示例数据约定，实际使用前先查询当前库的元信息。

主要配置参数说明：

| 参数 | 类型 | 说明 |
|------|------|------|
| `Name` | str | 因子库名称，默认 `"SQLDB"` |
| `DBType` | Literal | 当前表发现与连接支持 `"MySQL"`、`"PostgreSQL"`；参数模型中的其他类型尚未实现该路径 |
| `DBName` | str | 数据库名 |
| `IPAddr` | str | 数据库 IP 地址 |
| `Port` | int | 数据库端口 |
| `User` | str | 用户名 |
| `Pwd` | str | 密码 |
| `CharSet` | str | 字符集，如 `"utf8"` |
| `Connector` | str | 连接器：`"default"` / `"psycopg2"` / `"pymysql"` 等 |
| `TablePrefix` | str | 数据库表名前缀 |
| `InnerPrefix` | str | 内部前缀，默认 `"qs_"`，SQLDB 只识别该前缀的表 |
| `DTField` | str | 时点字段名，默认 `"datetime"` |
| `IDField` | str | ID 字段名，默认 `"code"` |
| `IgnoreFields` | List[str] | 忽略的字段名列表 |
| `CheckWriteData` | bool | 写入时是否校验数据一致性，默认 `False` |
| `CheckNullable` | bool | 写入时是否检查 NULL 约束，默认 `False` |
| `MetaTableName` | str | 元数据侧表名，默认 `"qs_meta"` |

In [ ]:
# 创建因子库对象并 connect
from QuantStudio.Factor.SQLDB import SQLDB

FDB = SQLDB().connect()

In [ ]:
print(qs_help(FDB))

In [ ]:
def read_table(fdb, table_name, factor_names, ids, dts, table_args=None):
    """读取调用方指定表和字段的因子面板。

    Args:
        fdb: 已连接的 SQLDB。
        table_name: 当前数据库中的表名。
        factor_names: 当前表中的因子名称列表。
        ids: 证券代码列表。
        dts: 请求日期列表。
        table_args: 表类型、维度字段和回溯参数。

    Returns:
        因子、日期、证券三个维度的 Panel。
    """
    table = fdb.getTable(table_name, args=table_args or {})
    return table.readData(factor_names=factor_names, ids=ids, dts=dts)

In [ ]:
# 获取因子库中的因子表列表
print(FDB.TableNames[:5])

# 因子表

In [ ]:
# 获取因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})

In [ ]:
print(qs_help(FT))

In [ ]:
# 因子列表
FT = FDB.getTable("stock_cn_day_bar")
print(FT.FactorNames)

## 表元信息

In [ ]:
# 获取因子表元信息的方法说明
print(qs_help(FT.getMetaData))

In [ ]:
# 获取因子表的所有元信息
print(FT.getMetaData())

## 时点序列

In [ ]:
# 获取时点序列的方法
print(qs_help(FT.getDateTime))

In [ ]:
# 给定起始时点和截止时点, 获取因子表的时点序列
FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 5))

## ID 序列

In [ ]:
# 获取因子表 ID 序列的方法
print(qs_help(FT.getID))

In [ ]:
# 获取因子表的 ID 序列
IDs = FT.getID()
print(IDs[:5], "..." if len(IDs)>5 else "", f"共 {len(IDs)} 个")

## 读取数据

In [ ]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
Data = FT.readData(factor_names=["close", "open"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

## 其他因子表类型

除了默认的 WideTable 外，SQLDB 还支持多种因子表类型，通过 `getTable` 的 `TableType` 参数指定。

### NarrowTable（窄表）

窄表将因子名和因子值分别存储在两个字段中，而不是每个因子占一列。适用于字段不固定的场景。

窄表需要指定：
- `FactorNameField`：存储因子名的字段
- `FactorValueField`：存储因子值的字段（默认取第一个非维度字段）
- `IDField` / `DTField`：ID 和时点字段

In [ ]:
# 窄表示例：stock_cn_factor_value_narrow 表
# 窄表需要指定 FactorNameField（因子名字段）和 FactorValueField（因子值字段）
NarrowFT = FDB.getTable("stock_cn_factor_value_narrow", args={
    "TableType": "NarrowTable",
    "FactorNameField": "factor_name",
    "FactorValueField": "factor_value",
    "LookBack": 0
})
print(f"\n因子列表: {NarrowFT.FactorNames}")

# 读取窄表数据
Data = NarrowFT.readData(factor_names=["pe", "pb"], ids=IDs, dts=DTs[:3])
print(f"\n窄表读取结果:\n{Data}")

In [ ]:
print(qs_help(NarrowFT))

### FeatureTable（特征表）

特征表没有时点维度，只有 ID 维度。适用于存储证券的静态属性，如行业分类、上市板块、公司全称等。

当 SQLDB 检测到表中没有时点字段时，会自动将表类型设为 `FeatureTable`。

In [ ]:
# 特征表示例：stock_cn_static_info 表存储了股票静态属性
FeatureFT = FDB.getTable("stock_cn_static_info", args={"TableType": "FeatureTable"})
print(f"因子列表: {FeatureFT.FactorNames}")

# 读取特征数据，dts 参数必须传但特征表会使用最大时点的数据
IDs = ["000001.SZ", "000002.SZ"]
Data = FeatureFT.readData(factor_names=["full_name", "listed_date"], ids=IDs, dts=[dt.datetime(2025, 1, 1)])
print("\n特征表数据:")
print(Data)

In [ ]:
print(qs_help(FeatureFT))

### TimeSeriesTable（时序表）

时序表没有 ID 维度，只有时点维度。适用于存储宏观指标，如无风险利率、汇率、GDP 等。

当 SQLDB 检测到表中没有 ID 字段时，会自动将表类型设为 `TimeSeriesTable`。

### MappingTable（映射表）

映射表用于存储 ID 之间的映射关系，如成分股与指数的从属关系。通常包含 `StartDT` 和 `EndDT` 字段标识映射的有效期。

In [ ]:
# 时序表示例：macro_indicator 存储宏观指标，无 ID 维度
TSFT = FDB.getTable("macro_indicator", args={"TableType": "TimeSeriesTable"})
print(f"因子列表: {TSFT.FactorNames}")

# 读取时序数据，ids 参数传空列表
MacroDTs = [dt.datetime(2020, 1, 1), dt.datetime(2020, 4, 1), dt.datetime(2020, 7, 1)]
Data = TSFT.readData(factor_names=["interest_rate", "cpi"], ids=[], dts=MacroDTs)
print(f"\n时序表数据:\n{Data}")

In [ ]:
print(qs_help(TSFT))

In [ ]:
# 映射表示例：stock_index_mapping 存储股票-指数映射关系
# 映射表 ID 字段名不是默认的 code，需要显式指定 IDField
MapFT = FDB.getTable("stock_index_mapping", args={
    "TableType": "MappingTable",
    "IDField": "stock_code"
})
print(f"因子列表: {MapFT.FactorNames}")

# 读取映射数据
Data = MapFT.readData(factor_names=["index_code", "start_date", "end_date"],
                       ids=["000001.SZ"], dts=[dt.datetime(2025, 1, 1)])
print(f"\n映射表数据:\n{Data}")

In [ ]:
print(qs_help(MapFT))

# 因子

In [ ]:
# 获取因子对象
F = FT.getFactor("close")

In [ ]:
print(qs_help(F))

## 因子元信息

In [ ]:
# 获取因子元信息的方法
print(qs_help(F.getMetaData))

In [ ]:
# 获取因子的所有元信息
print(F.getMetaData(key=None))

# 获取因子的数据类型
print(f"\n数据类型: {F.getMetaData(key='DataType')}")

## 因子时点序列

In [ ]:
# 获取因子时点序列的方法

# 获取因子的时点序列
print(f"\n时点序列: {F.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 5))}")

In [ ]:
print(qs_help(F.getDateTime))

## 因子 ID 序列

In [ ]:
# 获取因子 ID 序列的方法

# 获取因子的 ID 序列
IDs = F.getID()
print(f"\nID 序列: {IDs[:5]}... 共 {len(IDs)} 个")

In [ ]:
print(qs_help(F.getID))

## 读取数据

In [ ]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

# 因子库的变更操作

SQLDB 是支持写入和变更的因子库（继承自 `QuantStudio.Factor.FactorDB.WritableFactorDB`）。以下演示各种变更操作。

## 数据写入

`writeData` 方法将 Panel 数据写入因子库。`if_exists` 参数控制写入模式：

| 模式 | 说明 |
|------|------|
| `"update"` | 覆盖本次指定的日期与证券位置，未指定的位置保留 |

| `"append"` | 不覆盖原有数据，只在缺失位置填充新数据 |
| `"update_notnull"` | 仅用非空的新数据覆盖，空值保留原数据 |

In [ ]:
# 因子库数据写入方法
print(qs_help(FDB.writeData))

In [ ]:
# 数据写入示例
from QuantStudio.Core.QSObject import Panel

IDs = [str(i).zfill(6) + ".SZ" for i in range(1, 4)]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]

Data = Panel({
    "Factor1": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs),
    "Factor2": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs)
})
print("待写入的数据 : ")
print(Data)

print("-" * 60)
TargetTable = "TestTable"
FDB.writeData(data=Data, table_name=TargetTable, if_exists="update", data_type={"Factor1":"double", "Factor2":"double"})
print("写入后的因子表列表 : ")
print(FDB.TableNames)

print("-" * 60)
print("写入后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

## 创建表

`createTable` 方法用于在因子库中创建一张新的因子表。

In [ ]:
# 创建表的方法
print(qs_help(FDB.createTable))

## 添加因子

`addFactor` 方法用于向已有表中添加新的因子字段。如果表不存在，会自动创建该表。

In [ ]:
# 添加因子的方法
print(qs_help(FDB.addFactor))

## 设置表的元信息

In [ ]:
# 设置因子表元信息的方法
print(qs_help(FDB.setTableMetaData))

In [ ]:
# 设置表的元信息
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getMetaData())

print("-" * 60)
FDB.setTableMetaData(table_name=TargetTable, key="Description", value="这是一张测试表")
print("设置后的元信息 : ")
print(FT.getMetaData())

## 设置因子的元信息

In [ ]:
# 设置因子元信息的方法

# 设置因子的元信息
TargetTable = "TestTable"
TargetFactor = "Factor1"
FT = FDB.getTable(TargetTable)
print("\n设置前的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

print("-" * 60)
FDB.setFactorMetaData(table_name=TargetTable, ifactor_name=TargetFactor, key="Description", value="这是一个测试因子")
print("设置后的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

In [ ]:
print(qs_help(FDB.setFactorMetaData))

## 重命名因子

In [ ]:
# 重命名因子的方法

# 重命名因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print(f"\n重命名前因子列表 : {FT.FactorNames}")

FDB.renameFactor(table_name=TargetTable, old_factor_name="Factor1", new_factor_name="NewFactor1")
print(f"重命名后因子列表 : {FT.FactorNames}")

In [ ]:
print(qs_help(FDB.renameFactor))

## 删除因子

In [ ]:
# 删除因子的方法

# 删除因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print(f"\n删除前因子列表 : {FT.FactorNames}")

FDB.deleteFactor(table_name=TargetTable, factor_names=["NewFactor1"])
print(f"删除后因子列表 : {FT.FactorNames}")

In [ ]:
print(qs_help(FDB.deleteFactor))

## 重命名表

In [ ]:
# 重命名表的方法

# 重命名表
print(f"重命名前因子表 : {FDB.TableNames}")

FDB.renameTable(old_table_name="TestTable", new_table_name="NewTestTable")
print(f"重命名后因子表 : {FDB.TableNames}")

In [ ]:
print(qs_help(FDB.renameTable))

## 删除表

In [ ]:
# 删除表的方法

# 删除表
print(f"删除前因子表 : {FDB.TableNames}")

FDB.deleteTable(table_name="NewTestTable")
print(f"删除后因子表 : {FDB.TableNames}")

In [ ]:
print(qs_help(FDB.deleteTable))

## 删除数据

`deleteData` 方法用于删除表中指定时点或指定 ID 的数据。可以通过 `ids`、`dts` 或 `dt_ids`（时点+ID 组合）来精确控制删除范围。

In [ ]:
# 删除数据的方法
print(qs_help(FDB.deleteData))

## 元信息持久化

通过 `setTableMetaData()` 和 `setFactorMetaData()` 保存业务说明；重命名或删除对象时相关元信息同步更新。需要了解侧表结构、加载和同步规则时，参阅 [原理和细节](原理和细节.ipynb#sql-metadata)。

## 常见问题

- 表不存在：先查看 `TableNames`，确认连接的库、表前缀与元数据映射。
- 字段不存在：以当前表的 `FactorNames` 为准。
- 结果为空：核对日期范围、证券代码格式与公告时点；按业务含义设置回溯参数。
- 需要计算衍生因子：见 [因子开发](因子开发.ipynb)；缓存接入见 [缓存](../Core/缓存.ipynb)。